In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from peft import PeftModel
import torch

/Users/jekaterinasergejeva/Desktop/Masters/Lithuanian_LLM_Fine_Tuning/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1) Load fine-tuned model

In [7]:
# fine_tuned_model_path = "fine_tuned_tinyllama_model"
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
fine_tuned_model_path = "fine_tuned_gemma4_model"
model_name = "google/gemma-2b"

In [8]:
# quantization config (same as used for training)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [9]:
# 1. Load the original base model with quantization
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.bfloat16,
)

# 2. Load the PEFT adapter weights and apply them
# Assuming output_dir contains the adapter weights (adapter_model.safetensors, adapter_config.json)
loaded_model = PeftModel.from_pretrained(
    base_model,
    fine_tuned_model_path,
    dtype=torch.float16
)

loaded_tokenizer = AutoTokenizer.from_pretrained(fine_tuned_model_path)
loaded_tokenizer.pad_token = loaded_tokenizer.eos_token # Ensure tokenizer settings are consistent
loaded_tokenizer.padding_side = "right"

print("Model and tokenizer loaded successfully!")
print("Note: The model is loaded with 4-bit quantization and PEFT adapters applied.")

Loading weights: 100%|██████████| 164/164 [00:36<00:00,  4.48it/s]


Model and tokenizer loaded successfully!
Note: The model is loaded with 4-bit quantization and PEFT adapters applied.


### 2) Test fine-tuned model

In [ ]:
def generate(prompt, max_new_tokens=200):
    messages = [
        {"role": "user", "content": prompt}
    ]
    formatted_prompt = loaded_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = loaded_tokenizer(formatted_prompt, return_tensors="pt").to(loaded_model.device)

    with torch.no_grad():
        outputs = loaded_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return loaded_tokenizer.decode(new_tokens, skip_special_tokens=True)


prompt = "kas yra dirbtinis intelektas?"
print(generate(prompt))

('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');
('');



In [ ]:
prompt = "Kaip reikia ruoštis?"
print(generate(prompt))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Lietuvos sostinė - tai Lietuva, kurioje žemynais žaidina žvaigždžių daina ir šviesos kosmoso. Jis turi 101,000 km² teritoriją, kurios yra kelių miestų piktūros lūkesnys.


: 